In [1]:
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/deepchem/deepchem/master/datasets/delaney-processed.csv')
df.head()

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O)
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1


In [2]:
from rdkit import Chem
from rdkit.Chem import Descriptors

# Brought in the Lipinsky five descriptors to start
def get_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return pd.Series({
        'MolWt': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'TPSA': Descriptors.TPSA(mol),
    })

descriptors = df['smiles'].apply(get_descriptors)
df = pd.concat([df, descriptors], axis=1)
df.head()

,Compound ID,ESOL predicted log solubility in mols per litre,Minimum Degree,Molecular Weight,Number of H-Bond Donors,Number of Rings,Number of Rotatable Bonds,Polar Surface Area,measured log solubility in mols per litre,smiles,MolWt,LogP,NumHDonors,NumHAcceptors,TPSA
0,Amigdalin,-0.974,1,457.432,7,3,7,202.32,-0.77,OCC3OC(OCC2OC(OC(C#N)c1ccccc1)C(O)C(O)C2O)C(O)...,457.432,-3.10802,7.0,12.0,202.32
1,Fenfuram,-2.885,1,201.225,1,2,2,42.24,-3.30,Cc1occc1C(=O)Nc2ccccc2,201.225,2.84032,1.0,2.0,42.24
2,citral,-2.579,1,152.237,0,0,4,17.07,-2.06,CC(C)=CCCC(C)=CC(=O),152.237,2.87800,0.0,1.0,17.07
3,Picene,-6.618,2,278.354,0,5,0,0.00,-7.87,c1ccc2c(c1)ccc3c2ccc4c5ccccc5ccc43,278.354,6.29940,0.0,0.0,0.00
4,Thiophene,-2.232,2,84.143,0,1,0,0.00,-1.33,c1ccsc1,84.143,1.74810,0.0,1.0,0.00


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

X = df[['MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors', 'TPSA']]
y = df['measured log solubility in mols per litre']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
print(f"R²: {r2_score(y_test, predictions):.3f}")
print(f"RMSE: {root_mean_squared_error(y_test, predictions):.3f}")

R²: 0.851
RMSE: 0.841


In [4]:
from sklearn.linear_model import LinearRegression

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)

print(f"Linear Regression R²: {r2_score(y_test, lr_predictions):.3f}")
print(f"Linear Regression RMSE: {root_mean_squared_error(y_test, lr_predictions):.3f}")

Linear Regression R²: 0.740
Linear Regression RMSE: 1.109


In [5]:
importances = model.feature_importances_

for feature, importance in zip(X.columns, importances):
    print(f"{feature}: {importance:.3f}")

MolWt: 0.110
LogP: 0.824
NumHDonors: 0.008
NumHAcceptors: 0.012
TPSA: 0.046


In [6]:
from rdkit.Chem import Descriptors
print(len(Descriptors._descList))
for name, func in Descriptors._descList:
    print(name)

217
MaxAbsEStateIndex
MaxEStateIndex
MinAbsEStateIndex
MinEStateIndex
qed
SPS
MolWt
HeavyAtomMolWt
ExactMolWt
NumValenceElectrons
NumRadicalElectrons
MaxPartialCharge
MinPartialCharge
MaxAbsPartialCharge
MinAbsPartialCharge
FpDensityMorgan1
FpDensityMorgan2
FpDensityMorgan3
BCUT2D_MWHI
BCUT2D_MWLOW
BCUT2D_CHGHI
BCUT2D_CHGLO
BCUT2D_LOGPHI
BCUT2D_LOGPLOW
BCUT2D_MRHI
BCUT2D_MRLOW
AvgIpc
BalabanJ
BertzCT
Chi0
Chi0n
Chi0v
Chi1
Chi1n
Chi1v
Chi2n
Chi2v
Chi3n
Chi3v
Chi4n
Chi4v
HallKierAlpha
Ipc
Kappa1
Kappa2
Kappa3
LabuteASA
PEOE_VSA1
PEOE_VSA10
PEOE_VSA11
PEOE_VSA12
PEOE_VSA13
PEOE_VSA14
PEOE_VSA2
PEOE_VSA3
PEOE_VSA4
PEOE_VSA5
PEOE_VSA6
PEOE_VSA7
PEOE_VSA8
PEOE_VSA9
SMR_VSA1
SMR_VSA10
SMR_VSA2
SMR_VSA3
SMR_VSA4
SMR_VSA5
SMR_VSA6
SMR_VSA7
SMR_VSA8
SMR_VSA9
SlogP_VSA1
SlogP_VSA10
SlogP_VSA11
SlogP_VSA12
SlogP_VSA2
SlogP_VSA3
SlogP_VSA4
SlogP_VSA5
SlogP_VSA6
SlogP_VSA7
SlogP_VSA8
SlogP_VSA9
TPSA
EState_VSA1
EState_VSA10
EState_VSA11
EState_VSA2
EState_VSA3
EState_VSA4
EState_VSA5
EState_VSA6
ESt

In [8]:
def compute_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    mol_wt = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    h_donors = Descriptors.NumHDonors(mol)
    h_acceptors = Descriptors.NumHAcceptors(mol)
    tpsa = Descriptors.TPSA(mol)
    rotatable_bonds = Descriptors.NumRotatableBonds(mol)
    ring_count = Descriptors.RingCount(mol)
    heavy_atoms = Descriptors.HeavyAtomCount(mol)
    fraction_csp3 = Descriptors.FractionCSP3(mol)
    aromatic_atoms = sum(1 for atom in mol.GetAtoms() if atom.GetIsAromatic())
    aromatic_proportion = aromatic_atoms / heavy_atoms
    return pd.Series([mol_wt, logp, h_donors, h_acceptors, tpsa,
                       rotatable_bonds, ring_count, heavy_atoms,
                       fraction_csp3, aromatic_proportion])

df[['MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors', 'TPSA',
    'NumRotatableBonds', 'RingCount', 'HeavyAtomCount', 'FractionCSP3', 'AromaticProportion']] = df['smiles'].apply(compute_descriptors)

In [9]:
X = df[['MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors', 'TPSA',
        'NumRotatableBonds', 'RingCount', 'HeavyAtomCount', 'FractionCSP3', 'AromaticProportion']]
y = df['measured log solubility in mols per litre']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f"Random Forest (10 features) — R²: {r2_score(y_test, predictions):.3f}, RMSE: {root_mean_squared_error(y_test, predictions):.3f}")

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)
print(f"Linear Regression (10 features) — R²: {r2_score(y_test, lr_predictions):.3f}, RMSE: {root_mean_squared_error(y_test, lr_predictions):.3f}")

Random Forest (10 features) — R²: 0.867, RMSE: 0.794
Linear Regression (10 features) — R²: 0.758, RMSE: 1.069


In [10]:
importances = model.feature_importances_

for feature, importance in zip(X.columns, importances):
    print(f"{feature}: {importance:.3f}")

MolWt: 0.087
LogP: 0.808
NumHDonors: 0.006
NumHAcceptors: 0.009
TPSA: 0.035
NumRotatableBonds: 0.008
RingCount: 0.007
HeavyAtomCount: 0.018
FractionCSP3: 0.012
AromaticProportion: 0.010


In [11]:
new_descriptors = ['NumRotatableBonds', 'RingCount', 'HeavyAtomCount', 'FractionCSP3', 'AromaticProportion']
base_features = ['MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors', 'TPSA']

print("Baseline (5 features) — R²: 0.851, RMSE: 0.838\n")

for descriptor in new_descriptors:
    X_ablation = df[base_features + [descriptor]]
    X_train_a, X_test_a, y_train_a, y_test_a = train_test_split(X_ablation, y, test_size=0.2, random_state=42)

    model_a = RandomForestRegressor(n_estimators=100, random_state=42)
    model_a.fit(X_train_a, y_train_a)
    preds_a = model_a.predict(X_test_a)

    r2 = r2_score(y_test_a, preds_a)
    rmse = root_mean_squared_error(y_test_a, preds_a)
    print(f"+{descriptor}: R² = {r2:.3f}, RMSE = {rmse:.3f}")

Baseline (5 features) — R²: 0.851, RMSE: 0.838

+NumRotatableBonds: R² = 0.855, RMSE = 0.829
+RingCount: R² = 0.865, RMSE = 0.798
+HeavyAtomCount: R² = 0.856, RMSE = 0.825
+FractionCSP3: R² = 0.861, RMSE = 0.811
+AromaticProportion: R² = 0.859, RMSE = 0.815


In [12]:
X_logp_only = df[['LogP']]

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_logp_only, y, test_size=0.2, random_state=42)

model_l = RandomForestRegressor(n_estimators=100, random_state=42)
model_l.fit(X_train_l, y_train_l)
preds_l = model_l.predict(X_test_l)

print(f"LogP only — R²: {r2_score(y_test_l, preds_l):.3f}, RMSE: {root_mean_squared_error(y_test_l, preds_l):.3f}")

LogP only — R²: 0.696, RMSE: 1.199


In [13]:
X_two = df[['LogP', 'RingCount']]

X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(X_two, y, test_size=0.2, random_state=42)

model_2 = RandomForestRegressor(n_estimators=100, random_state=42)
model_2.fit(X_train_2, y_train_2)
preds_2 = model_2.predict(X_test_2)

print(f"LogP + RingCount — R²: {r2_score(y_test_2, preds_2):.3f}, RMSE: {root_mean_squared_error(y_test_2, preds_2):.3f}")

LogP + RingCount — R²: 0.781, RMSE: 1.016


In [14]:
X_three = df[['LogP', 'RingCount', 'MolWt']]

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_three, y, test_size=0.2, random_state=42)

model_3 = RandomForestRegressor(n_estimators=100, random_state=42)
model_3.fit(X_train_3, y_train_3)
preds_3 = model_3.predict(X_test_3)

print(f"LogP + RingCount + MolWt — R²: {r2_score(y_test_3, preds_3):.3f}, RMSE: {root_mean_squared_error(y_test_3, preds_3):.3f}")

LogP + RingCount + MolWt — R²: 0.856, RMSE: 0.826


In [15]:
X = df[['MolWt', 'LogP', 'NumHDonors', 'NumHAcceptors', 'TPSA',
        'NumRotatableBonds', 'RingCount', 'HeavyAtomCount', 'FractionCSP3', 'AromaticProportion']]
y = df['measured log solubility in mols per litre']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f"Random Forest (10 features) — R²: {r2_score(y_test, predictions):.3f}, RMSE: {root_mean_squared_error(y_test, predictions):.3f}")

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_predictions = lr_model.predict(X_test)
print(f"Linear Regression (10 features) — R²: {r2_score(y_test, lr_predictions):.3f}, RMSE: {root_mean_squared_error(y_test, lr_predictions):.3f}")

Random Forest (10 features) — R²: 0.867, RMSE: 0.794
Linear Regression (10 features) — R²: 0.758, RMSE: 1.069


In [16]:
from sklearn.model_selection import cross_val_score

cv_model = RandomForestRegressor(n_estimators=100, random_state=42)
cv_scores = cross_val_score(cv_model, X, y, cv=5, scoring='r2')

print(f"R² per fold: {cv_scores}")
print(f"Mean R²: {cv_scores.mean():.3f}")
print(f"Std R²: {cv_scores.std():.3f}")

R² per fold: [0.91666878 0.89941817 0.90477772 0.87868943 0.84942963]
Mean R²: 0.890
Std R²: 0.024


In [17]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_shuffled = cross_val_score(cv_model, X, y, cv=kf, scoring='r2')

print(f"R² per fold (shuffled): {cv_scores_shuffled}")
print(f"Mean R²: {cv_scores_shuffled.mean():.3f}")
print(f"Std R²: {cv_scores_shuffled.std():.3f}")

R² per fold (shuffled): [0.86504028 0.90839963 0.89052109 0.87840097 0.89994456]
Mean R²: 0.888
Std R²: 0.015


In [18]:
X_three = df[['LogP', 'RingCount', 'MolWt']]

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(X_three, y, test_size=0.2, random_state=42)

model_3 = RandomForestRegressor(n_estimators=100, random_state=42)
model_3.fit(X_train_3, y_train_3)
preds_3 = model_3.predict(X_test_3)

print(f"LogP + RingCount + MolWt — R²: {r2_score(y_test_3, preds_3):.3f}, RMSE: {root_mean_squared_error(y_test_3, preds_3):.3f}")

LogP + RingCount + MolWt — R²: 0.856, RMSE: 0.826


In [19]:
from sklearn.model_selection import cross_val_score

cv_model = RandomForestRegressor(n_estimators=100, random_state=42)
cv_scores = cross_val_score(cv_model, X, y, cv=5, scoring='r2')

print(f"R² per fold: {cv_scores}")
print(f"Mean R²: {cv_scores.mean():.3f}")
print(f"Std R²: {cv_scores.std():.3f}")

R² per fold: [0.91666878 0.89941817 0.90477772 0.87868943 0.84942963]
Mean R²: 0.890
Std R²: 0.024


In [20]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_shuffled = cross_val_score(cv_model, X, y, cv=kf, scoring='r2')

print(f"R² per fold (shuffled): {cv_scores_shuffled}")
print(f"Mean R²: {cv_scores_shuffled.mean():.3f}")
print(f"Std R²: {cv_scores_shuffled.std():.3f}")

R² per fold (shuffled): [0.86504028 0.90839963 0.89052109 0.87840097 0.89994456]
Mean R²: 0.888
Std R²: 0.015


In [21]:
X_three = df[['LogP', 'RingCount', 'MolWt']]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_model_3 = RandomForestRegressor(n_estimators=100, random_state=42)
cv_scores_3 = cross_val_score(cv_model_3, X_three, y, cv=kf, scoring='r2')

print(f"R² per fold (3-feature model): {cv_scores_3}")
print(f"Mean R²: {cv_scores_3.mean():.3f}")
print(f"Std R²: {cv_scores_3.std():.3f}")

R² per fold (3-feature model): [0.85482807 0.89217958 0.86438407 0.85832578 0.87468569]
Mean R²: 0.869
Std R²: 0.013
